In [0]:
from pyspark.sql.functions import col, when, current_date , year, input_file_name

silver_calendar_path = '/Volumes/workspace/default/filestore/tables/delta_silver/silver_calendar/'
df_calendar_silver = spark.read.format("delta").load(silver_calendar_path)


silver_products_path = '/Volumes/workspace/default/filestore/tables/delta_silver/silver_products/'
df_products_silver = spark.read.format("delta").load(silver_products_path)

silver_customers_path = '/Volumes/workspace/default/filestore/tables/delta_silver/silver_customers/'
df_customers_silver = spark.read.format("delta").load(silver_customers_path)


silver_stores_path = '/Volumes/workspace/default/filestore/tables/delta_silver/silver_stores/'
df_stores_silver = spark.read.format("delta").load(silver_stores_path)


silver_sales_path = '/Volumes/workspace/default/filestore/tables/delta_silver/silver_sales/'
df_sales_silver = spark.read.format("delta").load(silver_sales_path)



df_dim_calendar = (
    df_calendar_silver
        .withColumn(
            "month_name",
            when(col("month") == 1, "Enero")
            .when(col("month") == 2, "Febrero")
            .when(col("month") == 3, "Marzo")
            .when(col("month") == 4, "Abril")
            .when(col("month") == 5, "Mayo")
            .when(col("month") == 6, "Junio")
            .when(col("month") == 7, "Julio")
            .when(col("month") == 8, "Agosto")
            .when(col("month") == 9, "Setiembre")
            .when(col("month") == 10, "Octubre")
            .when(col("month") == 11, "Noviembre")
            .when(col("month") == 12, "Diciembre")
        )
        

)

df_dim_calendar.write.format("delta").mode("overwrite").save("/Volumes/workspace/default/filestore/tables/delta_gold/dim_calendar/")

df_dim_customers = (
    df_customers_silver
        .withColumn(
            "gender_customer",
            when(col("gender") == 'Male', "M")
            .when(col("gender") == 'Female', "F")
        )
        .withColumn("year_join", year(col("join_date")))
        .withColumn("ingestion_timestamp", current_date())
        .drop("gender","loyalty_member")
)

df_dim_customers.write.format("delta").mode("overwrite").option("mergeSchema", "true").save("/Volumes/workspace/default/filestore/tables/delta_gold/dim_customers/")


df_dim_products = (
    df_products_silver
    .withColumn("ingestion_timestamp", current_date())
    .drop("cocoa_percent")
)
df_dim_products.write.format("delta").mode("overwrite").option("mergeSchema", "true").save("/Volumes/workspace/default/filestore/tables/delta_gold/dim_products/")

df_dim_stores = (
    df_stores_silver
    .withColumn("ingestion_timestamp", current_date())
    .drop("city")
)

df_dim_stores.write.format("delta").mode("overwrite").option("mergeSchema", "true").save("/Volumes/workspace/default/filestore/tables/delta_gold/dim_stores/")



df_fact_sales = (
    df_sales_silver.alias("s")
        .join(
            df_dim_products.alias("p"),
            col("s.product_id") == col("p.product_id"),
            "left"
              )
        .join(
            df_dim_stores.alias("st"),
            col("s.store_id") == col("st.store_id"),
            "left"
        )
        .join(
            df_dim_customers.alias("c"),
            col("s.customer_id") == col("c.customer_id"),
            "left"
        )
        .join(
            df_dim_calendar.alias("cal"),
            col("s.order_date") == col("cal.date"),
            "left"
        )
        .select(
            col("s.order_id"),
            col("s.order_date"),
            col("cal.year"),
            col("cal.month_name"),
            col("p.product_id"),
            col("p.product_name"),
            col("p.brand"),
            col("p.category"),
            col("st.store_id"),
            col("st.store_name"),
            col("st.country"),
            col("st.store_type"),
            col("c.customer_id"),
            col("c.year_join"),
            col("c.gender_customer"),
            col("s.quantity"),
            col("s.unit_price"),
            col("s.discount"),
            col("s.revenue"),
            col("s.cost"),
            col("s.profit"),

        )
        )
        
df_fact_sales.write.format("delta").mode("overwrite").option("mergeSchema", "true").save("/Volumes/workspace/default/filestore/tables/delta_gold/fact_sales/")


display(df_fact_sales)



